# Run Presidio
ClinicalBERT de-identification transformer NER with spaCy (`en_core_web_sm`) tokenization (Plain Python)

## Set Up

### Import packages

In [ ]:
from pathlib import Path

import pandas as pd
from huggingface_hub import snapshot_download
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_analyzer.nlp_engine import (
    NerModelConfiguration,
    TransformersNlpEngine,
)
from transformers import AutoModelForTokenClassification, AutoTokenizer

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", 200)

### Helper functions

In [ ]:
def load_dataframe(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported input file type: {suffix}")


def save_dataframe(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df.to_csv(path, index=False)
        return
    if suffix in {".parquet", ".pq"}:
        df.to_parquet(path, index=False)
        return
    if suffix == ".jsonl":
        df.to_json(path, orient="records", lines=True)
        return
    if suffix == ".json":
        df.to_json(path, orient="records")
        return
    if suffix in {".pkl", ".pickle"}:
        df.to_pickle(path)
        return

    raise ValueError(f"Unsupported output file type: {suffix}")


def require_columns(df, columns):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")


def prepare_transformers_model(model_name, download_model=False):
    if not download_model:
        return

    snapshot_download(repo_id=model_name)
    AutoTokenizer.from_pretrained(model_name)
    AutoModelForTokenClassification.from_pretrained(model_name)


def build_transformers_analyzer(
    model_config,
    entity_mapping,
    labels_to_ignore,
    deny_list,
):
    ner_model_configuration = NerModelConfiguration(
        model_to_presidio_entity_mapping=entity_mapping,
        alignment_mode="expand",
        aggregation_strategy="max",
        labels_to_ignore=labels_to_ignore,
    )
    transformers_nlp_engine = TransformersNlpEngine(
        models=model_config,
        ner_model_configuration=ner_model_configuration,
    )
    analyzer = AnalyzerEngine(
        nlp_engine=transformers_nlp_engine,
        supported_languages=["en"],
    )

    if deny_list:
        pii_signal_recognizer = PatternRecognizer(
            supported_entity="PII_SIGNAL",
            deny_list=deny_list,
        )
        analyzer.registry.add_recognizer(pii_signal_recognizer)

    return analyzer


def entity_list_with_deny_list(entities, deny_list):
    entities_to_analyze = list(entities)
    if deny_list and "PII_SIGNAL" not in entities_to_analyze:
        entities_to_analyze.append("PII_SIGNAL")
    return entities_to_analyze


def analyze_one_text(analyzer, text, entities, allow_list):
    text = "" if pd.isna(text) else str(text)
    analyzer_results = analyzer.analyze(
        text=text,
        language="en",
        entities=entities,
        allow_list=allow_list,
    )

    detected_entities = ", ".join(
        f"{res.entity_type}: {res.start}-{res.end}" for res in analyzer_results
    )
    flagged_text = ", ".join(
        text[res.start : res.end] for res in analyzer_results
    )

    return {
        "detected_entities": detected_entities,
        "flagged_text": flagged_text,
    }


def run_presidio_analysis(df, target_col, analyzer, entities, allow_list):
    require_columns(df, [target_col])
    analysis_rows = [
        analyze_one_text(analyzer, text, entities, allow_list)
        for text in df[target_col]
    ]
    analysis_df = pd.DataFrame(analysis_rows)

    return pd.concat(
        [df.reset_index(drop=True), analysis_df.reset_index(drop=True)],
        axis=1,
    )

### Set parameters

See available Presidio entity types: https://microsoft.github.io/presidio/supported_entities/

In [ ]:
# Replace these paths and column names for your local data.
data_path = Path("data/model_inputs/all_eval_data.csv")
target_col = "text"
output_path = Path("data/model_outputs/clinicalbert_ner_results.csv")

data = load_dataframe(data_path)
require_columns(data, [target_col])

display(data.head())
print(f"Loaded {len(data):,} rows")

In [ ]:
my_entities_list = [
    "PERSON",
    "NRP",
    "DATE_TIME",
    "LOCATION",
    "EMAIL_ADDRESS",
    "PHONE_NUMBER",
    "US_SSN",
    "ORGANIZATION",
]
my_allow_list = []
my_deny_list = []

## Set Up ClinicalBERT Transformer NER

In [ ]:
transformers_model = "obi/deid_bert_i2b2"
download_transformers_model = False

prepare_transformers_model(
    transformers_model,
    download_model=download_transformers_model,
)

In [ ]:
# Transformer model config.
model_config = [
    {
        "lang_code": "en",
        "model_name": {
            "spacy": "en_core_web_sm",
            "transformers": transformers_model,
        },
    }
]

# Entity mappings between the model's labels and Presidio's labels.
mapping = {
    "PER": "PERSON",
    "LOC": "LOCATION",
    "ORG": "ORGANIZATION",
    "AGE": "AGE",
    "ID": "ID",
    "EMAIL": "EMAIL",
    "DATE": "DATE_TIME",
    "PHONE": "PHONE_NUMBER",
    "PERSON": "PERSON",
    "LOCATION": "LOCATION",
    "GPE": "LOCATION",
    "ORGANIZATION": "ORGANIZATION",
    "NORP": "NRP",
    "PATIENT": "PERSON",
    "STAFF": "PERSON",
    "HOSP": "LOCATION",
    "PATORG": "ORGANIZATION",
    "TIME": "DATE_TIME",
    "HCW": "PERSON",
    "HOSPITAL": "LOCATION",
    "FACILITY": "LOCATION",
    "VENDOR": "ORGANIZATION",
    "SSN": "US_SSN",
}

labels_to_ignore = ["OTHERPHI"]

## Run Presidio Analyzer

In [ ]:
analyzer = build_transformers_analyzer(
    model_config=model_config,
    entity_mapping=mapping,
    labels_to_ignore=labels_to_ignore,
    deny_list=my_deny_list,
)
entities_to_analyze = entity_list_with_deny_list(
    my_entities_list,
    my_deny_list,
)

results_df = run_presidio_analysis(
    data,
    target_col=target_col,
    analyzer=analyzer,
    entities=entities_to_analyze,
    allow_list=my_allow_list,
)

## Preview Results

In [ ]:
display(results_df.head())
print(f"Output row count: {len(results_df):,}")

## Save Results

In [ ]:
save_dataframe(results_df, output_path)
print(f"Saved {len(results_df):,} rows to {output_path.resolve()}")